# Chapter 12 - Part 1: Molecular graphs, tensors, and batches

[Chapter 11.4](Chapter11_Part4.ipynb) introduced a small graph neural network. This chapter builds its ingredients more carefully: which chemical information enters a graph, how variable-sized molecules become tensors, and how to check that batching preserves their identities.

**Why devote a chapter to GNNs?** Molecular graphs naturally accommodate different numbers of atoms and bonds. Shared local computations can learn representations of chemical neighborhoods and combine them into molecular predictions; geometric extensions also connect to 3D potential-energy models. These ideas make GNNs an important family to understand alongside fixed descriptors, fingerprints, and SMILES-based models. They are not a universal winner: performance depends on the data, target, representation, and evaluation protocol. See [Gilmer et al. (2017)](https://proceedings.mlr.press/v70/gilmer17a.html) for the message-passing framework and [Deng et al. (2023)](https://www.nature.com/articles/s41467-023-41948-6) for a systematic comparison of molecular prediction methods.

## Learning objectives

1. Describe node, edge, and graph-level data and state their tensor shapes.
2. Validate a SMILES record against an explicit chemical representation policy.
3. Encode categorical features with an unknown category and identify information the encoding loses.
4. Represent both directions of a bond and handle a graph with no edges.
5. Batch graphs by offsetting node indices, then pool atoms into the correct graph.
6. Verify atom-renumbering behavior and isolation between independent graphs.

**Execution:** run from the repository root in the [course environment](Readme.md). All examples are embedded, offline, and small; they use RDKit, NumPy, pandas, Matplotlib, and CPU PyTorch without a graph framework. No experimental labels or learned property predictions appear here. Outputs are written to `outputs/chapter12_part1/`.

**Chapter map:** [1. graphs and batching](Chapter12_Part1.ipynb) → [2. message passing and architectures](Chapter12_Part2.ipynb) → [3. measured-property learning](Chapter12_Part3.ipynb) → [4. diagnostics](Chapter12_Part4.ipynb) → [5. geometry and symmetry](Chapter12_Part5.ipynb).

**PyG extension:** [6. data and layers](Chapter12_Part6.ipynb) → [7. measured regression](Chapter12_Part7.ipynb) → [8. classification and explanations](Chapter12_Part8.ipynb).

### Start here: translate a familiar table into a graph

In Chapter 10, one row represented one molecule. A graph model first uses **one row per atom**, then combines atom rows into **one prediction per molecule**. A tensor is a numerical array; its *shape* lists how many entries it has along each axis. An embedding is a learned feature vector. A graph is a mathematical object here, not a plot.

For ethanol, `x` has three rows (C, C, O). An O–H bond is not an extra edge when hydrogens are implicit. `edge_index` stores four directed addresses for its two heavy-atom bonds. The graph target, if supplied, still has only one observation.

**First pass:** follow the molecule pictures, category table, batch diagram, and pooling example. Predict each shape before running its cell. **Deeper pass:** inspect the validator and renumbering tests; explain which real data mistake each catches. The [course guide](docs/course-guide.md) collects notation and prerequisite refreshers.

## 12.1.1. A graph is a representation of a chemical record

Write an attributed graph as $G=(V,E,X,A,U)$:

- $V$ is the set of atoms represented as nodes.
- $E$ describes which pairs are connected. An ordinary undirected bond is one edge in the chemical graph.
- $X$ stores atom attributes; $A$ here denotes **edge attributes**, not an adjacency matrix.
- $U$ contains information for the whole graph, such as net formal charge. Assay pH or temperature could also be graph-level inputs when known and relevant; we do not invent such values.

An atom index is a storage address, not a chemical feature. A bond graph usually contains no 3D conformation, intermolecular separation, solvent, or measurement protocol. Even an error-free graph conversion cannot establish that an assay label belongs to that chemical state.

We will store undirected chemical bonds as **two directed message edges**. With $N$ nodes, $B$ chemical bonds, $M=2B$ directed edges, $F$ atom features, and $D$ bond features:

| Tensor | Shape | Data type | Meaning |
|---|---|---|---|
| `x` | `(N, F)` | float32 | Atom feature rows |
| `edge_index` | `(2, M)` | int64 | Row 0: source; row 1: destination |
| `edge_attr` | `(M, D)` | float32 | One attribute row per directed edge |
| `u` | `(1, 2)` | float32 | Net formal charge and number of connected components |

Later, a batch adds `batch` (one graph index per node) and `ptr` (boundaries between graphs). We use edge lists for storage and a dense adjacency matrix only to visualize tiny examples. The memory for an edge list grows with the number of edges rather than $N^2$. The [message-passing framework](https://proceedings.mlr.press/v70/gilmer17a.html) explains how learning can operate on these attributed graphs.

In [ ]:
import os
for variable in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS']:
    os.environ[variable] = '1'
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch
from torch.nn import functional as F
from IPython.display import display, Image
from rdkit import Chem, rdBase
from rdkit.Chem import rdCIPLabeler
from rdkit.Chem.Draw import rdMolDraw2D

OUT = Path('outputs/chapter12_part1')
OUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)
DTYPE = torch.float32
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('RDKit:', rdBase.rdkitVersion, '| PyTorch:', torch.__version__, '| device: CPU')

## 12.1.2. Validation is a policy, not just a parser call

RDKit parsing and sanitization check syntax and rules such as permitted valence and aromaticity. A successful parse does not certify the intended structure, correct protonation at a given pH, or a physically stable compound. Conversely, a structure outside our lesson's policy may be valid chemistry.

Our declared policy is:

| Choice | This lesson's rule and consequence |
|---|---|
| Hydrogen representation | Heavy-atom nodes only. Attached implicit and bracket H counts become atom features. Actual `[H]` nodes are rejected; no hydrogens are silently deleted. `O` and `[NH4+]` each have one heavy-atom node. |
| Disconnected records | Reject by default; allow explicitly for a salt/component demonstration. No automatic largest-fragment selection. |
| Charge, isotopes, radicals | Preserve supplied values and encode them using declared categories. Radical-electron counts do not specify a molecule's total spin state. |
| Stereochemistry | Assign available atom and bond CIP labels. Unspecified stereo remains unspecified; we do not infer an enantiomer or an E/Z configuration. |
| Bond types | Support ordinary single, double, triple, and aromatic bonds. Reject other types here. In particular, a dative bond has chemical directionality that our symmetric two-edge attributes would erase. |
| Input format | Plain SMILES only; reject names/CXSMILES extensions, wildcard atoms, and explicit H nodes. Strip outer whitespace and clear atom-map labels, which are bookkeeping rather than model features. |
| Computational bounds | At most 128 heavy atoms and a bounded CIP assignment. Exceeding a bound raises an informative error. |

Keep an original record identifier and the rejection reason. Never drop failed structures and continue using an unchanged target array. The [RDKit reading guide](https://www.rdkit.org/docs/GettingStartedInPython.html#reading-single-molecules), [parser API](https://www.rdkit.org/docs/source/rdkit.Chem.rdmolfiles.html), and [RDKit Book](https://www.rdkit.org/docs/RDKit_Book.html#dative-bonds) document the underlying operations.

In [ ]:
SUPPORTED_BONDS = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
                   Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]

def prepare_molecule(mol, allow_disconnected=False):
    if mol is None or mol.GetNumAtoms() == 0:
        raise ValueError('No nonempty molecular graph was parsed.')
    mol = Chem.Mol(mol)  # Work on a copy, preserving the caller's molecule.
    if mol.GetNumAtoms() > 128:
        raise ValueError('Teaching limit exceeded: at most 128 heavy-atom nodes.')
    if any(atom.GetAtomicNum() == 0 for atom in mol.GetAtoms()):
        raise ValueError('Wildcard/dummy atoms are outside this molecular encoding.')
    if any(atom.GetAtomicNum() == 1 for atom in mol.GetAtoms()):
        raise ValueError('Explicit hydrogen nodes require a different hydrogen policy.')
    if not allow_disconnected and len(Chem.GetMolFrags(mol)) != 1:
        raise ValueError('Disconnected record: explicitly choose a component policy.')
    if any(bond.GetBondType() not in SUPPORTED_BONDS for bond in mol.GetBonds()):
        raise ValueError('Unsupported bond type; this encoder uses symmetric covalent edges.')
    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(0)
        if atom.HasProp('_CIPCode'):
            atom.ClearProp('_CIPCode')
    for bond in mol.GetBonds():
        if bond.HasProp('_CIPCode'):
            bond.ClearProp('_CIPCode')
    try:
        rdCIPLabeler.AssignCIPLabels(mol, maxRecursiveIterations=10000)
    except RuntimeError as error:
        raise ValueError('CIP assignment failed or exceeded the teaching bound.') from error
    return mol

def parse_record(smiles, allow_disconnected=False):
    if not isinstance(smiles, str) or not smiles.strip():
        raise ValueError('Expected a nonempty SMILES string.')
    options = Chem.SmilesParserParams()
    options.removeHs = False
    options.parseName = False
    options.allowCXSMILES = False
    with rdBase.BlockLogs():  # Reasons are recorded below instead of repeated parser messages.
        mol = Chem.MolFromSmiles(smiles.strip(), options)
    return prepare_molecule(mol, allow_disconnected=allow_disconnected)

In [ ]:
audit_inputs = [
    ('ethanol', 'CCO'), ('water', 'O'), ('ammonium', '[NH4+]'),
    ('sodium ion', '[Na+]'), ('13C methane', '[13CH4]'),
    ('lactic acid', 'C[C@H](O)C(=O)O'),
    ('salt', 'CC(=O)[O-].[Na+]'), ('bad valence', 'CO(C)C'),
    ('explicit H nodes', '[H]O[H]'), ('dative bond', 'N->[Cu]'),
    ('wildcard', '*C'), ('blank', '   '), ('missing', None),
]
audit_rows, accepted = [], {}
for record_id, (name, smiles) in enumerate(audit_inputs):
    try:
        mol = parse_record(smiles)
        accepted[name] = mol
        result = {'status': 'accepted', 'canonical_smiles': Chem.MolToSmiles(mol),
                  'reason': '', 'nodes': mol.GetNumAtoms(), 'bonds': mol.GetNumBonds()}
    except ValueError as error:
        result = {'status': 'rejected', 'canonical_smiles': None, 'reason': str(error),
                  'nodes': None, 'bonds': None}
    audit_rows.append({'record_id': record_id, 'name': name, 'input_smiles': smiles, **result})
audit = pd.DataFrame(audit_rows)
display(audit[['record_id', 'name', 'status', 'reason']])
audit.to_csv(OUT / 'input_audit.csv', index=False)
assert len(accepted) == 6 and len(audit) == 13

In [ ]:
drawer = rdMolDraw2D.MolDraw2DCairo(990, 500, 330, 250)
drawer.drawOptions().addAtomIndices = True
drawer.DrawMolecules(list(accepted.values()), legends=list(accepted))
drawer.FinishDrawing()
png = drawer.GetDrawingText()
(OUT / 'accepted_structures.png').write_bytes(png)
display(Image(data=png))
print('Displayed atom indices are addresses. They will not become node features.')

## 12.1.3. Categories are not continuous numbers

An element category is not a measurement to be scaled like temperature. We first map each category to an integer ID and then concatenate one-hot blocks. For example, category IDs 0, 1, and 2 do **not** mean the third element is twice the second. A learned embedding is another option; its lookup vocabulary must likewise be recorded with the model.

Each block reserves its **last position for `UNK`**. Missing from the vocabulary is different from zero and from padding. Sodium is deliberately outside our short element vocabulary, so it demonstrates the unknown path. Valid unseen elements can share `UNK`: that prevents an indexing failure but does not provide reliable extrapolation or a lossless representation. A research system may instead reject unsupported chemistry. The notebook reports unknown counts.

| Atom category / flag | Interpretation |
|---|---|
| Element | Atomic-number categories for a short declared vocabulary |
| Degree | Number of neighboring heavy-atom nodes, independent of bond order |
| Attached H count | Implicit plus bracket H counts; there are no H nodes in this encoding |
| Formal charge | Integer bookkeeping charge on that atom |
| Radical electrons | RDKit's assigned atom radical-electron count, not total spin multiplicity |
| Isotope | Specified mass number; 0 means no explicit isotope label, not zero mass |
| Atom CIP | R/S and r/s when assigned; otherwise unassigned or `UNK` |
| Aromatic / ring flags | RDKit's graph-based assignments; no 3D calculation |

`unassigned` deliberately does not distinguish an achiral atom from an unspecified stereocenter. Raw CW/CCW chiral tags depend on a neighbor-order convention, so we do not treat them as absolute atom properties. We use RDKit's CIP labeling implementation instead. Including CIP labels still does not guarantee complete representation of every stereochemical class, mixtures, or enhanced stereo groups. See [atom properties](https://www.rdkit.org/docs/source/rdkit.Chem.rdchem.html) and [CIP assignment](https://www.rdkit.org/docs/source/rdkit.Chem.rdCIPLabeler.html).

In [ ]:
ATOM_CATEGORIES = {
    'element': [6, 7, 8, 9, 15, 16, 17, 35, 53],
    'degree': [0, 1, 2, 3, 4],
    'attached_H': [0, 1, 2, 3, 4],
    'formal_charge': [-2, -1, 0, 1, 2],
    'radical_electrons': [0, 1, 2],
    'isotope': [0, 13, 15, 18],
    'atom_CIP': ['unassigned', 'R', 'S', 'r', 's'],
}
BOND_CATEGORIES = {
    'bond_type': ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC'],
    'bond_CIP': ['unassigned', 'E', 'Z'],
}
ATOM_FLAGS = ['aromatic', 'in_ring']
BOND_FLAGS = ['conjugated', 'in_ring']

def category_id(value, vocabulary):
    return vocabulary.index(value) if value in vocabulary else len(vocabulary)

def feature_names(schema, flags):
    return [f'{name}={value}' for name, values in schema.items() for value in [*values, 'UNK']] + flags

def one_hot_rows(rows, schema):
    ids = torch.tensor([[category_id(row[name], values) for name, values in schema.items()]
                        for row in rows], dtype=torch.long).reshape(len(rows), len(schema))
    blocks = [F.one_hot(ids[:, col], num_classes=len(values) + 1)
              for col, values in enumerate(schema.values())]
    return ids, torch.cat(blocks, dim=1).to(DTYPE)

NODE_FEATURES = feature_names(ATOM_CATEGORIES, ATOM_FLAGS)
EDGE_FEATURES = feature_names(BOND_CATEGORIES, BOND_FLAGS)
display(pd.DataFrame([{'block': name, 'known_categories': str(values),
                       'one_hot_width_including_UNK': len(values) + 1}
                      for name, values in ATOM_CATEGORIES.items()]))
print('Node feature width:', len(NODE_FEATURES), '| edge feature width:', len(EDGE_FEATURES))
assert category_id(11, ATOM_CATEGORIES['element']) == len(ATOM_CATEGORIES['element'])

### Build the tensors without an opaque featurizer

For each chemical bond between atoms $i$ and $j$, append $(i,j)$ and $(j,i)$ with the same symmetric bond attributes. The source and destination are tensor addresses; this duplication does not turn one covalent bond into two physical bonds. Conjugation, ring membership, and available E/Z CIP labels accompany the bond-type category.

Self-loops are not chemical bonds in this data structure. An architecture may later add a separate self-information route or explicitly marked self-loops. Those are modeling operations, not a modification of molecular connectivity. Edge order has no chemical meaning as long as `edge_index` and `edge_attr` are reordered together.

In [ ]:
def atom_values(atom):
    cip = atom.GetProp('_CIPCode') if atom.HasProp('_CIPCode') else (
        'unassigned' if atom.GetChiralTag() == Chem.ChiralType.CHI_UNSPECIFIED else 'specified_without_CIP')
    return {'element': atom.GetAtomicNum(), 'degree': atom.GetDegree(),
            'attached_H': atom.GetTotalNumHs(), 'formal_charge': atom.GetFormalCharge(),
            'radical_electrons': atom.GetNumRadicalElectrons(), 'isotope': atom.GetIsotope(),
            'atom_CIP': cip}

def bond_values(bond):
    cip = bond.GetProp('_CIPCode') if bond.HasProp('_CIPCode') else (
        'unassigned' if bond.GetStereo() == Chem.BondStereo.STEREONONE else 'specified_without_CIP')
    return {'bond_type': str(bond.GetBondType()), 'bond_CIP': cip}

def encode_molecule(mol, allow_disconnected=False):
    mol = prepare_molecule(mol, allow_disconnected=allow_disconnected)
    node_values = [atom_values(atom) for atom in mol.GetAtoms()]
    node_ids, node_one_hot = one_hot_rows(node_values, ATOM_CATEGORIES)
    node_flags = torch.tensor([[atom.GetIsAromatic(), atom.IsInRing()] for atom in mol.GetAtoms()], dtype=DTYPE)
    edges, edge_values, edge_flags = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges.extend([(i, j), (j, i)])
        edge_values.extend([bond_values(bond), bond_values(bond)])
        edge_flags.extend([[bond.GetIsConjugated(), bond.IsInRing()]] * 2)
    edge_ids, edge_one_hot = one_hot_rows(edge_values, BOND_CATEGORIES)
    graph = {
        'x': torch.cat([node_one_hot, node_flags], dim=1),
        'edge_index': torch.tensor(edges, dtype=torch.long).reshape(-1, 2).T.contiguous(),
        'edge_attr': torch.cat([edge_one_hot, torch.tensor(edge_flags, dtype=DTYPE).reshape(-1, 2)], dim=1),
        'u': torch.tensor([[Chem.GetFormalCharge(mol), len(Chem.GetMolFrags(mol))]], dtype=DTYPE),
        'node_category_ids': node_ids, 'edge_category_ids': edge_ids,
        'canonical_smiles': Chem.MolToSmiles(mol, isomericSmiles=True),
    }
    assert graph['x'].shape == (mol.GetNumAtoms(), len(NODE_FEATURES))
    assert graph['edge_index'].shape == (2, 2 * mol.GetNumBonds())
    assert graph['edge_attr'].shape == (2 * mol.GetNumBonds(), len(EDGE_FEATURES))
    assert torch.isfinite(graph['x']).all() and torch.isfinite(graph['edge_attr']).all()
    return graph

ethanol = accepted['ethanol']
ethanol_graph = encode_molecule(ethanol)
display(pd.DataFrame([atom_values(atom) for atom in ethanol.GetAtoms()]).rename_axis('atom_index'))
display(pd.DataFrame(ethanol_graph['edge_index'].T.numpy(), columns=['source', 'destination']).rename_axis('edge_row'))
print('x:', tuple(ethanol_graph['x'].shape), '| edge_index:', tuple(ethanol_graph['edge_index'].shape),
      '| edge_attr:', tuple(ethanol_graph['edge_attr'].shape), '| u:', ethanol_graph['u'].tolist())

In [ ]:
unknown_rows = []
for name, mol in accepted.items():
    graph = encode_molecule(mol)
    unknown = {feature: int((graph['node_category_ids'][:, column] == len(values)).sum())
               for column, (feature, values) in enumerate(ATOM_CATEGORIES.items())}
    unknown_rows.append({'record': name, **unknown})
display(pd.DataFrame(unknown_rows).set_index('record'))
sodium_graph = encode_molecule(accepted['sodium ion'])
assert sodium_graph['x'][0, NODE_FEATURES.index('element=UNK')] == 1
assert sodium_graph['u'][0, 0] == 1
carbon_graph = encode_molecule(parse_record('C'))
isotope_graph = encode_molecule(accepted['13C methane'])
assert not torch.equal(carbon_graph['x'], isotope_graph['x'])
print('Unknown element identity is visible; isotope-labeled methane differs from unspecified-isotope methane.')

## 12.1.4. Empty edges are not an empty molecule

Water written as `O` has one oxygen node, an attached-H count of two, and **zero heavy-atom bonds**. `[Na+]` also has one node and zero bonds, but different features and charge. An empty graph with zero atoms is a separate case and is rejected here.

The correct edge shapes are `(2, 0)` and `(0, D)`. A bare `torch.tensor([])` instead has shape `(0,)`, which breaks indexing or concatenation assumptions. The explicit `reshape` operations above retain all required axes. A batch consisting entirely of isolated atoms must work too.

In [ ]:
water_graph = encode_molecule(accepted['water'])
for name, graph in [('water', water_graph), ('sodium ion', sodium_graph)]:
    assert graph['x'].shape == (1, len(NODE_FEATURES))
    assert graph['edge_index'].shape == (2, 0)
    assert graph['edge_attr'].shape == (0, len(EDGE_FEATURES))
    source, destination = graph['edge_index']
    assert source.numel() == destination.numel() == 0
    print(name, ': x', tuple(graph['x'].shape), 'edge_index', tuple(graph['edge_index'].shape),
          'edge_attr', tuple(graph['edge_attr'].shape))
assert water_graph['x'][0, NODE_FEATURES.index('attached_H=2')] == 1

## 12.1.5. Stereochemistry: test a distinction that matters

The lactic-acid enantiomers below share connectivity but have different assigned atom CIP labels. The two difluoroethene structures differ in bond CIP labels. Our richer input can represent those differences; it does not prove that a future GNN will use them correctly or predict a stereosensitive endpoint. Stereochemical context and appropriate training labels are still required.

R/S and E/Z assignment is a graph calculation and can depend on information beyond a small local neighborhood. Including those labels as initial features supplies that information in advance. The lowercase r/s categories cover assigned pseudoasymmetric labels; we do not claim comprehensive coverage of every stereochemical convention. The [RDKit stereochemistry documentation](https://www.rdkit.org/docs/RDKit_Book.html#stereochemistry) explains relevant distinctions.

In [ ]:
lactic_pair = [parse_record(s) for s in ['C[C@H](O)C(=O)O', 'C[C@@H](O)C(=O)O']]
alkene_pair = [parse_record(s) for s in ['F/C=C/F', r'F/C=C\F']]
lactic_graphs = [encode_molecule(mol) for mol in lactic_pair]
alkene_graphs = [encode_molecule(mol) for mol in alkene_pair]
assert torch.equal(lactic_graphs[0]['edge_index'], lactic_graphs[1]['edge_index'])
assert not torch.equal(lactic_graphs[0]['x'], lactic_graphs[1]['x'])
assert not torch.equal(alkene_graphs[0]['edge_attr'], alkene_graphs[1]['edge_attr'])
display(pd.DataFrame([
    {'SMILES': Chem.MolToSmiles(mol),
     'atom_CIP_labels': [atom.GetProp('_CIPCode') for atom in mol.GetAtoms() if atom.HasProp('_CIPCode')],
     'bond_CIP_labels': [bond.GetProp('_CIPCode') for bond in mol.GetBonds() if bond.HasProp('_CIPCode')]}
    for mol in [*lactic_pair, *alkene_pair]
]))

## 12.1.6. Build a disjoint batch

Suppose the graphs contain $n_0,n_1,\ldots,n_{K-1}$ nodes. Concatenate node rows, offset each graph's edge indices by its starting node position, and concatenate edge attributes in the same order.

$$\mathrm{ptr}=[0,n_0,n_0+n_1,\ldots,\textstyle\sum_g n_g].$$

For a node at global row $i$, `batch[i]` gives its graph number. Graph $g$ occupies rows `ptr[g]:ptr[g+1]`. These identifiers route computation; feeding a graph's position in the batch as a chemical feature would be a mistake.

| Batched tensor | Shape |
|---|---|
| `x` | `(sum(N_g), F)` |
| `edge_index`, `edge_attr` | `(2, sum(M_g))`, `(sum(M_g), D)` |
| `batch` | `(sum(N_g),)` |
| `ptr` | `(K + 1,)` |
| `u` | `(K, 2)` |

**Invariant to enforce:** `batch[source] == batch[destination]` for every edge. Sharing parameters across molecules is intended; creating an edge between independent records is not. All-isolated batches still have nodes, `batch`, `ptr`, and `u`, even though they have no edge rows.

In [ ]:
def validate_batch(packed):
    n, width = packed['x'].shape
    k = packed['u'].shape[0]
    assert n > 0 and k > 0 and width == len(NODE_FEATURES)
    assert packed['edge_index'].dtype == packed['batch'].dtype == packed['ptr'].dtype == torch.long
    assert packed['edge_index'].shape[0] == 2
    m = packed['edge_index'].shape[1]
    assert packed['edge_attr'].shape == (m, len(EDGE_FEATURES))
    assert packed['batch'].shape == (n,) and packed['ptr'].shape == (k + 1,)
    assert packed['ptr'][0] == 0 and packed['ptr'][-1] == n
    counts = packed['ptr'][1:] - packed['ptr'][:-1]
    assert torch.all(counts > 0)
    expected_membership = torch.repeat_interleave(torch.arange(k), counts)
    assert torch.equal(packed['batch'], expected_membership)
    if m:
        assert packed['edge_index'].min() >= 0 and packed['edge_index'].max() < n
        source, destination = packed['edge_index']
        assert torch.equal(packed['batch'][source], packed['batch'][destination]), 'Cross-graph edge detected.'
    assert packed['u'].shape == (k, 2)
    return packed

def pack_graphs(graphs):
    if not graphs or any(graph['x'].shape[0] == 0 for graph in graphs):
        raise ValueError('Supply at least one nonempty encoded graph.')
    counts = [graph['x'].shape[0] for graph in graphs]
    ptr = torch.tensor([0, *np.cumsum(counts).tolist()], dtype=torch.long)
    packed = {
        'x': torch.cat([graph['x'] for graph in graphs], dim=0),
        'edge_index': torch.cat([graph['edge_index'] + ptr[g] for g, graph in enumerate(graphs)], dim=1),
        'edge_attr': torch.cat([graph['edge_attr'] for graph in graphs], dim=0),
        'u': torch.cat([graph['u'] for graph in graphs], dim=0),
        'batch': torch.repeat_interleave(torch.arange(len(graphs)), torch.tensor(counts)),
        'ptr': ptr,
    }
    return validate_batch(packed)

batch_names = ['ethanol', 'water', 'acetate', 'sodium ion']
batch_mols = [ethanol, accepted['water'], parse_record('CC(=O)[O-]'), accepted['sodium ion']]
graphs = [encode_molecule(mol) for mol in batch_mols]
packed = pack_graphs(graphs)
isolated_batch = pack_graphs([water_graph, sodium_graph])
assert isolated_batch['edge_index'].shape == (2, 0)
assert isolated_batch['edge_attr'].shape == (0, len(EDGE_FEATURES))
print('ptr:', packed['ptr'].tolist(), '| batch:', packed['batch'].tolist())
display(pd.DataFrame({'graph': batch_names, 'start': packed['ptr'][:-1].tolist(),
                      'stop_exclusive': packed['ptr'][1:].tolist(),
                      'formal_charge': packed['u'][:, 0].tolist(),
                      'components': packed['u'][:, 1].tolist()}))

In [ ]:
def adjacency(graph):
    # Destination rows, source columns: (adjacency @ x)[i] sums features arriving at i.
    matrix = torch.zeros((graph['x'].shape[0], graph['x'].shape[0]), dtype=DTYPE)
    source, destination = graph['edge_index']
    matrix[destination, source] = 1
    return matrix

matrix = adjacency(packed).numpy()
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.8), layout='constrained',
                         gridspec_kw={'width_ratios': [1.2, 1]})
axes[0].imshow(matrix, cmap='Greys', vmin=0, vmax=1)
colors = ['#267fa3', '#c78924', '#a1548f', '#3e9570']
for g, (start, stop) in enumerate(zip(packed['ptr'][:-1], packed['ptr'][1:])):
    start, stop = int(start), int(stop)
    axes[0].add_patch(Rectangle((start - 0.5, start - 0.5), stop - start, stop - start,
                                fill=False, edgecolor=colors[g], linewidth=2))
axes[0].set_xticks(range(matrix.shape[0]))
axes[0].set_yticks(range(matrix.shape[0]))
axes[0].set(xlabel='Global source node', ylabel='Global destination node', title='Block-disjoint adjacency')
for g, name in enumerate(batch_names):
    start, stop = packed['ptr'][g:g+2].tolist()
    axes[1].barh(g, stop - start, left=start, color=colors[g], height=0.65)
    axes[1].text((start + stop)/2, g, f'[{start}:{stop}]', color='white', ha='center', va='center', fontsize=9)
axes[1].set_yticks(range(len(batch_names)), batch_names)
axes[1].invert_yaxis()
axes[1].set(xlabel='Position in concatenated node tensor', xlim=(0, len(packed['batch'])),
            title='ptr defines slices; batch defines membership')
fig.savefig(OUT / 'batch_layout.png')
plt.show()

## 12.1.7. Pool by graph index, not across the whole batch

Given node vectors $h_i$, graph-level sum and mean readouts are

$$s_g=\sum_{i:\,\mathrm{batch}[i]=g}h_i,\qquad
\bar h_g=\frac{s_g}{N_g}.$$

`index_add` routes each row into the graph identified by `batch`. Sum pooling can retain counts; mean pooling normalizes them by node count. Neither choice alone guarantees a physically extensive or intensive prediction. A later nonlinear readout or bias can change additivity.

We pool the raw one-hot features below. The carbon and oxygen columns then give exact heavy-atom counts under our vocabulary. This is a deterministic bookkeeping calculation, **not a trained GNN or a benchmark**. Source: [PyTorch indexed addition](https://docs.pytorch.org/docs/2.11/generated/torch.Tensor.index_add_.html).

In [ ]:
def graph_pool(node_vectors, batch_data, reduce='sum'):
    if node_vectors.ndim != 2 or node_vectors.shape[0] != batch_data['x'].shape[0]:
        raise ValueError('Expected one feature row per node in this batch.')
    k = batch_data['u'].shape[0]
    pooled = node_vectors.new_zeros((k, node_vectors.shape[1]))
    pooled = pooled.index_add(0, batch_data['batch'], node_vectors)
    if reduce == 'sum':
        return pooled
    if reduce == 'mean':
        counts = (batch_data['ptr'][1:] - batch_data['ptr'][:-1]).to(node_vectors.dtype)
        return pooled / counts[:, None]
    raise ValueError('Use sum or mean pooling.')

summed = graph_pool(packed['x'], packed)
mean = graph_pool(packed['x'], packed, reduce='mean')
carbon_column = NODE_FEATURES.index('element=6')
oxygen_column = NODE_FEATURES.index('element=8')
display(pd.DataFrame({'graph': batch_names,
                      'carbon_count': summed[:, carbon_column].tolist(),
                      'oxygen_count': summed[:, oxygen_column].tolist(),
                      'oxygen_fraction_of_nodes': mean[:, oxygen_column].tolist()}))
expected_oxygen = torch.tensor([sum(atom.GetAtomicNum() == 8 for atom in mol.GetAtoms())
                               for mol in batch_mols], dtype=DTYPE)
torch.testing.assert_close(summed[:, oxygen_column], expected_oxygen)

### Worked research check: a feature file with the right shape can still be wrong

A collaborator sends a new batch with the same number of feature columns. They have accidentally exchanged the carbon and oxygen columns. Shape checks pass, but a saved model would interpret the atoms incorrectly. Before any prediction, compare a quantity with an independent chemical meaning: sum the oxygen indicators and compare with RDKit's atom count.

The left panel catches this deliberate corruption. The right panel separates **count** from **fraction**: mean pooling describes composition and does not preserve size by itself. These are calculated properties of the supplied records, not experimental labels. In a research pipeline, store and compare the full ordered feature schema as well as these small reference-molecule checks.

In [ ]:
swapped_x = packed['x'].clone()
swapped_x[:, [carbon_column, oxygen_column]] = packed['x'][:, [oxygen_column, carbon_column]]
swapped_oxygen = graph_pool(swapped_x, packed)[:, oxygen_column]
assert swapped_x.shape == packed['x'].shape
assert not torch.equal(swapped_oxygen, expected_oxygen)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), layout='constrained')
positions = np.arange(len(batch_names))
axes[0].bar(positions-.18, expected_oxygen.numpy(), .36, label='Chemical oxygen count')
axes[0].bar(positions+.18, swapped_oxygen.numpy(), .36, label='Read after C/O swap')
axes[0].set(ylabel='Count', ylim=(0,2.7), title='Same tensor shape; changed meaning')
axes[0].legend(fontsize=8)
axes[1].bar(positions, mean[:, oxygen_column].numpy(), color='#319795')
axes[1].set(ylabel='Fraction of represented atoms that are O', ylim=(0,1.12),
            title='Mean pooling answers a different question')
for ax in axes:
    ax.set_xticks(positions, batch_names, rotation=15)
fig.savefig(OUT / 'feature_schema_research_check.png', dpi=150)
plt.show()
print('Caught the intentional feature-order error; the original tensors remain intact.')

### A salt record is not a batch of two separate observations

`CC(=O)[O-].[Na+]` denotes disconnected components in **one supplied record**. If explicitly retained, it has one graph-level target and one readout row, with net formal charge zero and two components. A batch containing acetate and sodium as separate records has two readout rows and two graph-level metadata rows.

No bond messages pass between the salt's components either. Pooling them together does not model ionic interactions, dissociation equilibria, or the geometry of an ion pair. The number of targets and the component policy must follow the scientific endpoint. Similar care applies to mixtures, solvates, and reaction participants.

Two copies of an identical component illustrate another limitation: sum pooling doubles its vector, whereas mean pooling alone cannot distinguish one copy from two. Our component-count metadata could supply that distinction if a later model uses it.

In [ ]:
salt_mol = parse_record('CC(=O)[O-].[Na+]', allow_disconnected=True)
salt_graph = encode_molecule(salt_mol, allow_disconnected=True)
salt_batch = pack_graphs([salt_graph])
separate_components = pack_graphs(graphs[2:])
assert salt_batch['u'].tolist() == [[0.0, 2.0]]
assert separate_components['u'].tolist() == [[-1.0, 1.0], [1.0, 1.0]]
print('One salt record: readout shape', tuple(graph_pool(salt_batch['x'], salt_batch).shape))
print('Two independent records: readout shape', tuple(graph_pool(separate_components['x'], separate_components).shape))

one_copy = pack_graphs([ethanol_graph])
two_copy_mol = parse_record('CCO.CCO', allow_disconnected=True)
two_copies = pack_graphs([encode_molecule(two_copy_mol, allow_disconnected=True)])
torch.testing.assert_close(graph_pool(two_copies['x'], two_copies), 2 * graph_pool(one_copy['x'], one_copy))
torch.testing.assert_close(graph_pool(two_copies['x'], two_copies, 'mean'),
                           graph_pool(one_copy['x'], one_copy, 'mean'))
print('Sum doubles for two identical components; mean remains the same.')

## 12.1.8. Test what should change under atom renumbering

`Chem.RenumberAtoms(mol, order)` makes new atom $i$ correspond to old atom `order[i]`. Therefore:

- node feature rows should reorder in exactly that way;
- edge addresses should change consistently, with attributes still attached to the same chemical bonds;
- a graph-level symmetric readout should stay unchanged.

The whole node matrix is **equivariant**: it reorders with the nodes. The graph readout is **invariant**: it stays the same. Literal equality of serialized `edge_index` arrays is not the criterion because edge order is arbitrary. For these tiny tests, we compare an edge-attribute tensor indexed by destination and source after applying the permutation. Production batching can stay sparse.

A canonical SMILES string can be useful for identity auditing, but it is not required to make graph computation invariant. The test below intentionally uses the noncanonical RDKit atom order. See [RDKit atom renumbering](https://www.rdkit.org/docs/source/rdkit.Chem.rdmolops.html).

In [ ]:
def dense_edge_attributes(graph):
    n = graph['x'].shape[0]
    dense = torch.zeros((n, n, len(EDGE_FEATURES)), dtype=DTYPE)
    source, destination = graph['edge_index']
    dense[destination, source] = graph['edge_attr']
    return dense

permutation_rows = []
for mol in [ethanol, lactic_pair[0], alkene_pair[0], accepted['water']]:
    n = mol.GetNumAtoms()
    order = list(reversed(range(n)))
    original = encode_molecule(mol)
    changed = encode_molecule(Chem.RenumberAtoms(mol, order))
    torch.testing.assert_close(changed['x'], original['x'][order], rtol=0, atol=0)
    old_edges = dense_edge_attributes(original)
    torch.testing.assert_close(dense_edge_attributes(changed), old_edges[order][:, order], rtol=0, atol=0)
    original_batch, changed_batch = pack_graphs([original]), pack_graphs([changed])
    torch.testing.assert_close(graph_pool(changed_batch['x'], changed_batch),
                               graph_pool(original_batch['x'], original_batch), rtol=0, atol=0)
    permutation_rows.append({'SMILES': Chem.MolToSmiles(mol), 'new_to_old_order': order,
                              'node_and_edge_equivariance': True, 'readout_invariance': True})
display(pd.DataFrame(permutation_rows))

def index_weighted_atomic_number(mol):
    return sum((i + 1) * atom.GetAtomicNum() for i, atom in enumerate(mol.GetAtoms()))
renumbered_ethanol = Chem.RenumberAtoms(ethanol, [2, 1, 0])
assert index_weighted_atomic_number(ethanol) != index_weighted_atomic_number(renumbered_ethanol)
print('An index-weighted sum changes:', index_weighted_atomic_number(ethanol),
      'to', index_weighted_atomic_number(renumbered_ethanol), 'without changing the molecule.')

## 12.1.9. Batch isolation needs a behavioral check too

The absence of cross-graph edges is necessary but not sufficient for every possible model to be independent of batch composition. For example, subtracting a mean calculated across **all nodes in the batch** can make one graph's output depend on its batch companions. A graph-specific operation has different semantics. Training-time normalization layers also need deliberate treatment.

Use a transparent, fixed local operation here: add the features of immediate neighbors to each node's own vector, then pool by graph. This introduces the source/destination aggregation that [Part 2](Chapter12_Part2.ipynb) will turn into learned messages. There are no trainable weights in this check.

We verify that separate processing equals batched processing, reversing the graph order only reverses outputs, and changing another graph leaves the first graph's result unchanged. Then we deliberately corrupt one edge and confirm that validation catches it.

In [ ]:
def local_summary(batch_data):
    source, destination = batch_data['edge_index']
    neighbor_sum = torch.zeros_like(batch_data['x']).index_add(0, destination, batch_data['x'][source])
    return graph_pool(batch_data['x'] + neighbor_sum, batch_data)

joint_summary = local_summary(packed)
separate_summary = torch.cat([local_summary(pack_graphs([graph])) for graph in graphs], dim=0)
torch.testing.assert_close(joint_summary, separate_summary, rtol=0, atol=0)
torch.testing.assert_close(local_summary(pack_graphs(list(reversed(graphs)))), joint_summary.flip(0), rtol=0, atol=0)
changed_companion = pack_graphs([graphs[0], encode_molecule(parse_record('c1ccccc1'))])
torch.testing.assert_close(local_summary(changed_companion)[0], joint_summary[0], rtol=0, atol=0)
assert torch.isfinite(local_summary(isolated_batch)).all()

corrupted = {name: value.clone() for name, value in packed.items()}
corrupted['edge_index'][1, 0] = packed['ptr'][1]  # First edge now enters the water record.
try:
    validate_batch(corrupted)
except AssertionError as error:
    print('Expected rejection:', error)
else:
    raise AssertionError('The cross-graph edge should have been rejected.')

def globally_centered_readout(batch_data):
    centered = batch_data['x'] - batch_data['x'].mean(dim=0, keepdim=True)
    return graph_pool(centered, batch_data)

alone = globally_centered_readout(pack_graphs([graphs[0]]))[0]
with_companion = globally_centered_readout(changed_companion)[0]
assert not torch.allclose(alone, with_companion)
print('Local batching checks passed. Centering over the whole batch changes the first graph:',
      float(torch.linalg.vector_norm(alone - with_companion)))

## 12.1.10. What must travel with a graph model?

Save the exact category order and unknown convention, feature names, hydrogen/component/stereo policy, tensor conventions, software version, and target/context definition. The same tensor width with a different column order is a different model input. Keep original record identifiers for dataset auditing; they are not predictive features.

The fixed vocabulary here was declared before looking at data. If you instead learn a vocabulary, a scaler, an imputer, or feature selection from a dataset, fit it on training data only. Keep alternate SMILES, conformers, and duplicate records for one compound together in evaluation splits. New scaffold or assay generalization needs an appropriate grouping protocol, not just valid tensor shapes.

**Limits:** these tensors omit conformer coordinates, solvent, conditions, total spin state, and some stereochemical detail. Unknown categories merge distinct values. Atom and bond assignments follow a toolkit's chemical conventions. Passing all checks below establishes the tested representation and routing behavior; it does not establish a useful molecular prediction.

In [ ]:
schema = {
    'schema_version': 1, 'rdkit_version': rdBase.rdkitVersion, 'torch_version': str(torch.__version__),
    'atom_categories': ATOM_CATEGORIES, 'bond_categories': BOND_CATEGORIES,
    'unknown_rule': 'last category in every one-hot block',
    'node_features': NODE_FEATURES, 'edge_features': EDGE_FEATURES,
    'edge_convention': 'edge_index[0] source, edge_index[1] destination; two directions per supported bond',
    'global_features': ['net_formal_charge', 'connected_components'],
    'hydrogens': 'heavy-atom nodes; attached implicit/bracket H counts; reject explicit H nodes',
    'components': 'reject disconnected by default; retain all components only by explicit choice',
    'stereo': 'bounded rdCIPLabeler assignment; atom R/S/r/s and bond E/Z; unspecified remains unassigned',
    'scope': 'representation and batching checks only; no fitted model or experimental target',
    'checks': ['empty-edge shapes', 'unknown-category reporting', 'stereo distinctions',
               'node/edge permutation equivariance', 'readout invariance', 'separate/batch equivalence',
               'graph-order equivariance', 'companion independence', 'cross-graph edge rejection'],
}
(OUT / 'graph_schema.json').write_text(json.dumps(schema, indent=2) + '\n', encoding='utf-8')
torch.save({key: value for key, value in packed.items()}, OUT / 'example_batch.pt')
print('Saved schema, audited records, figures, and the tiny example batch to', OUT)

## Exercises

1. Ethanol has three heavy atoms and two bonds. State `x`, `edge_index`, and `edge_attr` shapes using this notebook's feature widths. Why are there four directed edges?
2. For node counts `[3, 1, 4]`, write `ptr` and `batch`. Where does local edge `0 → 2` of graph 2 go in the concatenated tensor?
3. Explain why `O`, `[NH4+]`, and `[Na+]` all have empty heavy-atom edge lists but must not receive identical features.
4. Is `UNK` proof that an input is invalid? Can two distinct unseen elements become indistinguishable under that part of the encoding?
5. Why is a two-component salt different from a batch of two independent observations even when neither representation contains an intercomponent bond?
6. Why are R/S labels safer as absolute atom features than using raw CW/CCW tags without accounting for neighbor order? What information remains missing when stereo is unspecified?
7. Permute the edge columns and permute `edge_attr` rows by the same order. Should `local_summary` change? What if only one tensor is permuted in a bond-aware model?
8. A program passes the cross-edge assertion but subtracts a whole-batch mean from node vectors. Why can a molecule's prediction depend on what else is in the batch?
9. Two identical disconnected components have the same mean-pooled raw vector as one copy. Does that prove that every model using mean pooling must output the same answer?
10. You want to model an ion-pair interaction energy. Which necessary information is missing from a disconnected bond graph, and why does successful batching not solve the problem?

<details><summary>Selected solutions</summary>

1. Read the printed widths: `x=(3,F)`, `edge_index=(2,4)`, and `edge_attr=(4,D)`. Each atom can receive its bonded neighbor's message; physical bond count remains two.
2. `ptr=[0,3,4,8]`; `batch=[0,0,0,1,2,2,2,2]`. Graph 2 starts at 4, so the edge becomes `4 → 6`.
3. Water has an oxygen with two attached H atoms; ammonium has nitrogen, four H atoms, and charge +1; sodium has a different element category and no H atoms. No-edge is a connectivity statement, not chemical identity.
4. No. It means outside this declared vocabulary. Such values can collide, so flag them and define a domain policy rather than claiming learned transfer.
5. The former has one target/readout and graph metadata for the complete record; the latter has two. Neither encodes an interaction merely by having disconnected components.
6. CW/CCW tags refer to a neighbor-order convention. CIP assignment resolves priority-based labels where applicable; unspecified configuration cannot be recovered from an absent stereo specification.
7. Consistent edge reordering should not change a symmetric aggregate. Reordering only the attributes can attach one bond's properties to a different bond, changing the meaning of the input. Floating-point summation can introduce tiny order-dependent numerical differences in learned models.
8. The preprocessing statistic contains nodes from other molecules even though the edge list is disjoint. Test outputs with different batch companions and choose operations whose semantics match the task.
9. No. Component counts or other global inputs can distinguish them, and a different architecture can encode size explicitly. The equality concerns that particular pooled vector.
10. The graph gives no ion separation or orientation, solvent, dielectric conditions, or specified electronic state. A task-appropriate geometric/physical model and validation are needed.

</details>

**Continue:** [Part 2 - message passing and architectures](Chapter12_Part2.ipynb). The architectural formulas work with different feature widths; compare each notebook's declared encoding rather than assuming its columns are interchangeable.